# PlantCLEF 2015 S-CNN(A) Genus Figures

Builds and displays draft-ready genus-level diagnostic plots for the trained VGG16 S-CNN(A) checkpoint.


## 1. Runtime Check


In [ ]:
import torch

print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Clone Or Update Project


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

PROJECT_DIR = Path('/content/diploma')
REPO_URL = 'https://github.com/robodanill/diploma.git'
BRANCH = 'robodanill/main'


def clone_project():
    os.chdir('/content')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)


def pull_project() -> bool:
    if not (PROJECT_DIR / '.git').exists():
        return False
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
    return result.returncode == 0


if PROJECT_DIR.exists():
    print(f'Trying to update existing project: {PROJECT_DIR}')
    if not pull_project():
        print('Pull failed or project is not a git repository; cloning a fresh copy.')
        clone_project()
else:
    print(f'Project not found at {PROJECT_DIR}; cloning a fresh copy.')
    clone_project()

os.chdir(PROJECT_DIR)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.[ml]'], check=True)

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
print(f'Project commit: {commit}')


## 3. Mount Google Drive


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 4. Restore LeafScan Data And Paper60 Metadata


In [ ]:
%%bash
set -euo pipefail
trap 'echo "FAILED at line $LINENO: $BASH_COMMAND" >&2' ERR
export PYTHONUNBUFFERED=1
cd /content/diploma

ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz
TEST_ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz

echo "checking required archives"
ls -lh /content/drive/MyDrive/PlantCLEF2015*.tar.gz 2>/dev/null || true
if [ ! -f "$ARCHIVE" ]; then
  echo "Missing LeafScan training archive: $ARCHIVE" >&2
  exit 2
fi
if [ ! -f "$TEST_ARCHIVE" ]; then
  echo "Missing LeafScan test archive: $TEST_ARCHIVE" >&2
  exit 3
fi

rm -rf data/plantclef2015
mkdir -p data/plantclef2015

echo "extracting training archive: $ARCHIVE"
tar -xzf "$ARCHIVE" -C data/plantclef2015
test -f data/plantclef2015/leafscan/metadata.csv
cp data/plantclef2015/leafscan/metadata.csv data/plantclef2015/leafscan_metadata.csv

echo "extracting test archive: $TEST_ARCHIVE"
rm -rf data/plantclef2015/test_leafscan
mkdir -p data/plantclef2015/test_leafscan
tar -xzf "$TEST_ARCHIVE" -C data/plantclef2015/test_leafscan
test -f data/plantclef2015/test_leafscan/leafscan/metadata.csv
cp data/plantclef2015/test_leafscan/leafscan/metadata.csv data/plantclef2015/test_leafscan_metadata.csv

python - <<'PY2'
import csv
from collections import Counter, defaultdict
from pathlib import Path
from PIL import Image

with open('data/plantclef2015/leafscan_metadata.csv', newline='', encoding='utf-8') as file:
    source_rows = list(csv.DictReader(file))
with open('data/plantclef2015/test_leafscan_metadata.csv', newline='', encoding='utf-8') as file:
    test_rows = list(csv.DictReader(file))

test_species = {row['species'] for row in test_rows}
paper60_rows = [row for row in source_rows if row['species'] in test_species]
source_species = {row['species'] for row in source_rows}
missing_in_train = sorted(test_species - source_species)
if missing_in_train:
    raise RuntimeError(f'Missing test species in train metadata: {missing_in_train}')

fieldnames = list(source_rows[0].keys())
with open('data/plantclef2015/leafscan_paper60_metadata.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(paper60_rows)

train_count_by_species = Counter(row['species'] for row in paper60_rows)
underfilled_species = sorted(species for species in test_species if train_count_by_species[species] < 6)
if underfilled_species:
    print('paper60 species with fewer than 6 train images:', underfilled_species)
    rows_by_species = defaultdict(list)
    for row in paper60_rows:
        rows_by_species[row['species']].append(row)
    augmented_dir = Path('data/plantclef2015/leafscan/augmented')
    augmented_dir.mkdir(parents=True, exist_ok=True)
    leafscan_root = Path('data/plantclef2015/leafscan')
    angles = [180, 90, 270, 15, -15]
    augmented_rows = []
    for species in underfilled_species:
        species_rows = rows_by_species[species]
        if not species_rows:
            raise RuntimeError(f'Cannot augment {species}: no train rows found')
        needed = 6 - len(species_rows)
        for index in range(needed):
            base_row = species_rows[index % len(species_rows)]
            source_path = Path(base_row['image_path'])
            if not source_path.is_absolute():
                source_path = leafscan_root / source_path
            angle = angles[index % len(angles)]
            output_name = f"{source_path.stem}_aug_rot{angle}_{index + 1}.jpg".replace('-', 'm')
            output_path = augmented_dir / output_name
            with Image.open(source_path) as image:
                image.convert('RGB').rotate(angle, expand=True, fillcolor=(255, 255, 255)).save(output_path, quality=95)
            augmented_row = dict(base_row)
            augmented_row['image_path'] = str(output_path.relative_to(leafscan_root))
            if 'source_xml' in augmented_row:
                augmented_row['source_xml'] = f"{augmented_row['source_xml']}#aug_rot{angle}"
            augmented_rows.append(augmented_row)
    paper60_rows.extend(augmented_rows)
    with open('data/plantclef2015/leafscan_paper60_metadata.csv', 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(paper60_rows)
    print('paper60 augmented rows added:', len(augmented_rows))

train_count_by_species = Counter(row['species'] for row in paper60_rows)
underfilled_species = sorted(species for species in test_species if train_count_by_species[species] < 6)
if underfilled_species:
    raise RuntimeError(f'Cannot build 6-shot paper subset: {underfilled_species}')

print('leafscan source rows:', len(source_rows))
print('paper60 train rows:', len(paper60_rows))
print('paper60 train genera:', len({row['genus'] for row in paper60_rows}))
print('paper60 train species:', len({row['species'] for row in paper60_rows}))
print('paper60 test rows:', len(test_rows))
print('paper60 test genera:', len({row['genus'] for row in test_rows}))
print('paper60 test species:', len(test_species))
print('paper60 six-shot training rows:', 6 * len(test_species))
PY2


## 5. Locate The Genus Checkpoint On Drive


In [ ]:
from pathlib import Path
import shutil as shutil_module

MANUAL_GENUS_CHECKPOINT = ''
DRIVE_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/diploma_checkpoints')
LOCAL_CHECKPOINT = Path('/content/diploma/checkpoints/scnn_genus_vgg16_best.pt')

if MANUAL_GENUS_CHECKPOINT:
    checkpoint = Path(MANUAL_GENUS_CHECKPOINT)
else:
    patterns = [
        'leafscan_vgg16/**/scnn_genus_vgg16_best.pt',
        '**/scnn_genus_vgg16_best.pt',
        'leafscan_vgg16/**/scnn_genus_vgg16.pt',
        '**/scnn_genus_vgg16.pt',
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(DRIVE_CHECKPOINT_ROOT.glob(pattern))
    candidates = sorted(set(candidates), key=lambda path: path.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f'No VGG16 genus checkpoint found under {DRIVE_CHECKPOINT_ROOT}')
    checkpoint = candidates[0]

if not checkpoint.exists():
    raise FileNotFoundError(checkpoint)

LOCAL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
shutil_module.copy2(checkpoint, LOCAL_CHECKPOINT)
Path('/content/diploma/.genus_checkpoint_path').write_text(str(LOCAL_CHECKPOINT), encoding='utf-8')
print('Selected Drive checkpoint:', checkpoint)
print('Copied to:', LOCAL_CHECKPOINT)
print('Size:', LOCAL_CHECKPOINT.stat().st_size)


## 6. Save Genus Draft Figures


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

CHECKPOINT="$(cat .genus_checkpoint_path)"
BASE_OUT="/content/drive/MyDrive/diploma_diagnostics"
STAMP="$(date -u +%Y%m%dT%H%M%SZ)"

for MODE in comparator l1; do
  OUT_DIR="$BASE_OUT/genus_vgg16_${MODE}_${STAMP}"
  mkdir -p "$OUT_DIR"
  echo "=== genus artifacts score_mode=$MODE ==="
  python -u -m plant_classifier.training.eval_genus_cli \
    --config configs/leafscan_paper60_training.yaml \
    --query-config configs/leafscan_test.yaml \
    --checkpoint "$CHECKPOINT" \
    --max-species 0 \
    --references-per-genus 6 \
    --reference-level genus \
    --reference-seed 42 \
    --reference-split train \
    --score-mode "$MODE" \
    --output-dir "$OUT_DIR" \
    --top-k 5 15 30 50
  ls -lh "$OUT_DIR"
done


## 7. Display Genus Draft Figures Inline


In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

BASE = Path('/content/drive/MyDrive/diploma_diagnostics')


def latest_dir(pattern: str) -> Path | None:
    candidates = sorted(BASE.glob(pattern), key=lambda path: path.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None


def display_png(path: Path, width: int = 900) -> None:
    if not path.exists():
        display(Markdown(f'`{path}` not found'))
        return
    display(Markdown(f'`{path}`'))
    display(Image(filename=str(path), width=width))

for title, pattern in [
    ('Genus comparator artifacts', 'genus_vgg16_comparator_*'),
    ('Genus L1 artifacts', 'genus_vgg16_l1_*'),
]:
    display(Markdown(f'### {title}'))
    directory = latest_dir(pattern)
    if directory is None:
        display(Markdown('No artifact directory found yet.'))
        continue
    display(Markdown(f'Directory: `{directory}`'))
    for filename in [
        'topk_genus_accuracy.png',
        'genus_confusion_matrix.png',
        'genus_rank_histogram.png',
        'per_genus_top1_accuracy.png',
    ]:
        display_png(directory / filename)


## 8. Серия рамок 2x2 по ошибкам родов

Ячейка строит 10 компактных PNG-рамок из пар изображений, которые модель путала на уровне рода. Внутри самой рамки нет отметок правильности: только изображения и короткая подпись с родом. Для более убедительных иллюстраций из ближайших ошибочных кандидатов выбирается визуально наиболее похожий эталон.


In [ ]:
from pathlib import Path
from collections import Counter
from functools import lru_cache
from textwrap import wrap
import os
import sys

import numpy as np
import torch
import yaml
from IPython.display import Image as IPyImage, Markdown, display
from PIL import Image, ImageDraw, ImageFont

PROJECT_DIR = Path('/content/diploma')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))

from plant_classifier.data import filter_records_by_split, limit_records_by_species, load_metadata_csv  # noqa: E402
from plant_classifier.models.siamese import BackboneSpec, build_siamese_network  # noqa: E402
from plant_classifier.preprocessing.views import LeafBoundingBoxCrop  # noqa: E402
from plant_classifier.training.genus_eval import embed_image, rank_references, select_reference_records  # noqa: E402
from plant_classifier.training.image_pairs import build_image_transform  # noqa: E402

INTEREST_GENERA = ['Lippia', 'Prunus', 'Fraxinus']
REFERENCE_SEED = 42
GENUS_SCORE_MODE = 'l1'
REFERENCES_PER_GENUS = 6
FRAME_COUNT = 10
PAIRS_PER_FRAME = 2
WRONG_CANDIDATES_TO_COMPARE = 12
OUTPUT_DIR = Path('/content/drive/MyDrive/diploma_diagnostics/figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if hasattr(Image, 'Resampling'):
    RESAMPLE = Image.Resampling.LANCZOS
else:
    RESAMPLE = getattr(Image, 'LANCZOS', Image.BICUBIC)


def require_existing_path(path: Path, hint: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(f'{path} не найден. {hint}')
    return path


def load_config(path: Path) -> dict:
    require_existing_path(path, 'Проверьте, что репозиторий обновлен и выполнены подготовительные ячейки.')
    with path.open('r', encoding='utf-8') as file:
        return yaml.safe_load(file)


def load_records(dataset_config: dict):
    metadata_path = require_existing_path(
        Path(dataset_config['metadata']),
        'Выполните секцию 4: Restore LeafScan Data And Paper60 Metadata.',
    )
    dataset_root = require_existing_path(
        Path(dataset_config['root']),
        'Выполните секцию 4: Restore LeafScan Data And Paper60 Metadata.',
    )
    return load_metadata_csv(
        metadata_path=metadata_path,
        dataset_root=dataset_root,
        image_column=dataset_config['image_column'],
        family_column=dataset_config['family_column'],
        genus_column=dataset_config['genus_column'],
        species_column=dataset_config['species_column'],
    )


def apply_subset(records, dataset_config: dict):
    subset = dataset_config.get('subset')
    if not subset:
        return records
    return limit_records_by_species(
        records,
        max_species=subset.get('max_species'),
        min_images_per_species=int(subset.get('min_images_per_species', 1)),
        max_images_per_species=subset.get('max_images_per_species'),
        seed=subset.get('seed'),
    )


def preprocessing_enabled(config: dict) -> bool:
    preprocessing = config.get('preprocessing', {})
    return bool(preprocessing.get('enabled', preprocessing.get('leaf_bbox', False)))


@torch.inference_mode()
def embed_image_cpu(model, image_path: Path, transform, device: torch.device):
    return embed_image(model, image_path, transform, device).detach().cpu()


def release_cuda_model(model) -> None:
    try:
        model.cpu()
    except Exception:
        pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


leaf_crop = LeafBoundingBoxCrop(padding=10)


@lru_cache(maxsize=4096)
def load_leaf_crop(path_string: str) -> Image.Image:
    with Image.open(path_string) as image:
        rgb = image.convert('RGB')
    try:
        return leaf_crop(rgb)
    except Exception:
        return rgb


@lru_cache(maxsize=4096)
def visual_descriptor(path_string: str) -> np.ndarray:
    image = load_leaf_crop(path_string).convert('L')
    side = max(image.size)
    square = Image.new('L', (side, side), 255)
    square.paste(image, ((side - image.width) // 2, (side - image.height) // 2))
    square = square.resize((64, 64), RESAMPLE)
    vector = 255.0 - np.array(square, dtype=np.float32).reshape(-1)
    vector = vector - vector.mean()
    norm = np.linalg.norm(vector)
    if norm < 1e-6:
        return vector
    return vector / norm


def visual_similarity(left_path: Path, right_path: Path) -> float:
    left = visual_descriptor(str(left_path))
    right = visual_descriptor(str(right_path))
    return float(np.dot(left, right))


def short_species_name(species: str) -> str:
    parts = species.replace('×', 'x').split()
    if len(parts) >= 3 and parts[1].lower() == 'x':
        return ' '.join(parts[:3])
    return ' '.join(parts[:2]) if len(parts) >= 2 else species


def load_demo_font(size: int):
    for candidate in [
        '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',
    ]:
        try:
            return ImageFont.truetype(candidate, size)
        except OSError:
            pass
    return ImageFont.load_default()


def fit_leaf_image(path: Path, size: tuple[int, int]) -> Image.Image:
    image = load_leaf_crop(str(path)).copy()
    image.thumbnail(size, RESAMPLE)
    canvas = Image.new('RGB', size, (255, 255, 255))
    canvas.paste(image, ((size[0] - image.width) // 2, (size[1] - image.height) // 2))
    return canvas


def text_bbox(draw: ImageDraw.ImageDraw, text: str, font):
    if hasattr(draw, 'textbbox'):
        return draw.textbbox((0, 0), text, font=font)
    width, height = draw.textsize(text, font=font)
    return (0, 0, width, height)


def draw_centered_label(draw: ImageDraw.ImageDraw, box: tuple[int, int, int, int], text: str, font) -> None:
    x0, y0, x1, y1 = box
    lines = (wrap(text, width=27) or [text])[:2]
    spacing = 4
    line_boxes = [text_bbox(draw, line, font) for line in lines]
    widths = [bbox[2] - bbox[0] for bbox in line_boxes]
    heights = [bbox[3] - bbox[1] for bbox in line_boxes]
    total_height = sum(heights) + spacing * max(0, len(lines) - 1)
    y = y0 + (y1 - y0 - total_height) / 2
    for line, width, height in zip(lines, widths, heights):
        draw.text((x0 + (x1 - x0 - width) / 2, y), line, font=font, fill=(15, 23, 42))
        y += height + spacing


def render_demo_frames(
    pairs: list[dict],
    *,
    label_getter,
    output_prefix: str,
    frame_count: int = FRAME_COUNT,
    display_width: int = 760,
) -> list[Path]:
    selected = pairs[: frame_count * PAIRS_PER_FRAME]
    if len(selected) < PAIRS_PER_FRAME:
        raise RuntimeError('Недостаточно пар для построения рамок')

    scale = 2
    tile_w = 300 * scale
    image_h = 282 * scale
    label_h = 38 * scale
    margin = 10 * scale
    gap = 8 * scale
    border = 1 * scale
    panel_h = image_h + label_h
    canvas_w = margin * 2 + tile_w * 2 + gap
    canvas_h = margin * 2 + panel_h * PAIRS_PER_FRAME + gap * (PAIRS_PER_FRAME - 1)
    label_font = load_demo_font(20 * scale)
    paths: list[Path] = []

    for frame_index in range(frame_count):
        chunk = selected[frame_index * PAIRS_PER_FRAME : (frame_index + 1) * PAIRS_PER_FRAME]
        if len(chunk) < PAIRS_PER_FRAME:
            break
        canvas = Image.new('RGB', (canvas_w, canvas_h), (255, 255, 255))
        draw = ImageDraw.Draw(canvas)
        for row, pair in enumerate(chunk):
            for col, record in enumerate([pair['left'], pair['right']]):
                x0 = margin + col * (tile_w + gap)
                y0 = margin + row * (panel_h + gap)
                draw.rectangle(
                    (x0, y0, x0 + tile_w, y0 + panel_h),
                    fill=(250, 250, 250),
                    outline=(203, 213, 225),
                    width=border,
                )
                image = fit_leaf_image(record.image_path, (tile_w - 16 * scale, image_h - 12 * scale))
                canvas.paste(image, (x0 + 8 * scale, y0 + 6 * scale))
                draw_centered_label(
                    draw,
                    (x0 + 6 * scale, y0 + image_h, x0 + tile_w - 6 * scale, y0 + panel_h),
                    label_getter(record),
                    label_font,
                )
        output_path = OUTPUT_DIR / f'{output_prefix}_{frame_index + 1:02d}.png'
        canvas.save(output_path, quality=98, dpi=(220, 220))
        paths.append(output_path)

    display(Markdown(f'Создано рамок: **{len(paths)}**. Папка: `{OUTPUT_DIR}`'))
    for output_path in paths:
        display(Markdown(f'`{output_path}`'))
        display(IPyImage(filename=str(output_path), width=display_width))
    return paths


def choose_demo_pairs(
    candidates: list[dict],
    target_count: int,
    label_key: str,
    priority_getter=None,
) -> list[dict]:
    if not candidates:
        return []
    selected: list[dict] = []
    used_queries: set[str] = set()
    per_label: Counter[str] = Counter()

    def add_candidate(candidate: dict, max_per_label: int = 4) -> bool:
        query_id = str(candidate['left'].image_path)
        label = getattr(candidate['left'], label_key)
        if query_id in used_queries or per_label[label] >= max_per_label:
            return False
        selected.append(candidate)
        used_queries.add(query_id)
        per_label[label] += 1
        return True

    for genus in INTEREST_GENERA:
        genus_candidates = [
            item for item in candidates
            if item['left'].genus == genus or item['right'].genus == genus
        ]
        genus_candidates = sorted(genus_candidates, key=lambda item: item['visual_similarity'], reverse=True)
        for candidate in genus_candidates[:2]:
            if len(selected) >= target_count:
                break
            add_candidate(candidate, max_per_label=3)

    def sort_key(item: dict):
        extra_priority = priority_getter(item) if priority_getter else ()
        if not isinstance(extra_priority, tuple):
            extra_priority = (extra_priority,)
        return (
            *extra_priority,
            0 if item['left'].genus in INTEREST_GENERA or item['right'].genus in INTEREST_GENERA else 1,
            -item['visual_similarity'],
        )

    ranked = sorted(candidates, key=sort_key)
    for candidate in ranked:
        if len(selected) >= target_count:
            break
        add_candidate(candidate)

    for candidate in ranked:
        if len(selected) >= target_count:
            break
        query_id = str(candidate['left'].image_path)
        if query_id not in used_queries:
            selected.append(candidate)
            used_queries.add(query_id)

    return selected[:target_count]


config = load_config(PROJECT_DIR / 'configs/leafscan_paper60_training.yaml')
query_config = load_config(PROJECT_DIR / 'configs/leafscan_test.yaml')
checkpoint_marker = require_existing_path(
    PROJECT_DIR / '.genus_checkpoint_path',
    'Выполните секцию 5: Locate The Genus Checkpoint On Drive.',
)
genus_checkpoint = require_existing_path(
    Path(checkpoint_marker.read_text(encoding='utf-8').strip()),
    'Выполните секцию 5: Locate The Genus Checkpoint On Drive.',
)

records = apply_subset(load_records(config['dataset']), config['dataset'])
query_records = load_records(query_config['dataset'])
query_species = {record.species for record in query_records}
reference_pool = [
    record for record in filter_records_by_split(records, 'train')
    if record.species in query_species
]
references = select_reference_records(
    reference_pool,
    taxonomic_level='genus',
    references_per_label=REFERENCES_PER_GENUS,
    seed=REFERENCE_SEED,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_siamese_network(BackboneSpec(name=config['model']['backbone'], pretrained=False)).to(device)
model.load_state_dict(torch.load(genus_checkpoint, map_location=device))
model.eval()
transform = build_image_transform(
    'global',
    image_size=int(config['views']['global']['image_size']),
    crop_size=int(config['views']['local']['crop_size']),
    preprocessing=preprocessing_enabled(config),
)
reference_embeddings = [(record, embed_image_cpu(model, record.image_path, transform, device)) for record in references]

pair_candidates = []
for query in query_records:
    query_embedding = embed_image_cpu(model, query.image_path, transform, device)
    ranked = rank_references(model, query_embedding, reference_embeddings, score_mode=GENUS_SCORE_MODE)
    wrong_ranked = [(record, score) for record, score in ranked if record.genus != query.genus]
    if not wrong_ranked:
        continue
    top_same_rank = next(
        (index for index, (record, _score) in enumerate(ranked, start=1) if record.genus == query.genus),
        None,
    )
    if top_same_rank == 1:
        continue
    visual_candidates = wrong_ranked[:WRONG_CANDIDATES_TO_COMPARE]
    reference, _score = max(
        visual_candidates,
        key=lambda item: visual_similarity(query.image_path, item[0].image_path),
    )
    pair_candidates.append(
        {
            'left': query,
            'right': reference,
            'visual_similarity': visual_similarity(query.image_path, reference.image_path),
        }
    )

release_cuda_model(model)
del model

selected_pairs = choose_demo_pairs(pair_candidates, FRAME_COUNT * PAIRS_PER_FRAME, label_key='genus')
print(f'Подобрано пар по родам: {len(selected_pairs)} из {len(pair_candidates)} кандидатов')
genus_demo_frame_paths = render_demo_frames(
    selected_pairs,
    label_getter=lambda record: f'Род: {record.genus}',
    output_prefix='genus_demo_pairs_lippia_prunus_fraxinus',
)


## 9. Серия рамок 2x2 по ошибкам видов

Отдельная ячейка для видового уровня. Она использует сохраненные веса S-CNN(A) и S-CNN(B), выбирает случаи, где итоговый top-1 вид отличается от истинного, и сохраняет такие же компактные рамки. Внутри рамок показаны только короткие названия видов, без отметок правильности.


In [ ]:
if 'render_demo_frames' not in globals():
    raise RuntimeError('Сначала выполните ячейку 8: она загружает данные и общие функции для рамок.')

from collections import defaultdict

from plant_classifier.inference.scnn import TwoStageSiamesePredictor  # noqa: E402
from plant_classifier.training.genus_eval import select_genus_references  # noqa: E402
from plant_classifier.training.species_eval import build_reference_embeddings, select_species_references  # noqa: E402

SPECIES_REFERENCE_SEED = 42
SPECIES_GENUS_CANDIDATES = 15
SPECIES_GENUS_SCORE_MODE = 'l1'
SPECIES_SCORE_MODE = 'l1'
SPECIES_AGGREGATION = 'mean'
SPECIES_REFERENCES_PER_LABEL = 6
SPECIES_FRAME_COUNT = 10
SPECIES_CHECKPOINT_OVERRIDE = ''


def locate_species_checkpoint() -> Path:
    if SPECIES_CHECKPOINT_OVERRIDE:
        manual = Path(SPECIES_CHECKPOINT_OVERRIDE)
        if not manual.exists():
            raise FileNotFoundError(f'Указанный checkpoint вида не найден: {manual}')
        return manual

    marker = PROJECT_DIR / '.species_checkpoint_path'
    if marker.exists():
        marked = Path(marker.read_text(encoding='utf-8').strip())
        if marked.exists():
            return marked

    search_root = Path('/content/drive/MyDrive/diploma_checkpoints')
    patterns = [
        'leafscan_vgg16/**/scnn_species_vgg16_best.pt',
        '**/scnn_species_vgg16_best.pt',
        'leafscan_vgg16/**/scnn_species_vgg16.pt',
        '**/scnn_species_vgg16.pt',
    ]
    matches = []
    for pattern in patterns:
        matches.extend(path for path in search_root.glob(pattern) if path.is_file())
    if not matches:
        raise FileNotFoundError(
            'Не найден checkpoint S-CNN(B) на Google Drive. '
            'Проверьте /content/drive/MyDrive/diploma_checkpoints или заполните SPECIES_CHECKPOINT_OVERRIDE.'
        )
    newest = max(matches, key=lambda path: path.stat().st_mtime)
    marker.write_text(str(newest), encoding='utf-8')
    return newest


species_checkpoint = locate_species_checkpoint()
print(f'Используется checkpoint S-CNN(B): {species_checkpoint}')

config = load_config(PROJECT_DIR / 'configs/leafscan_paper60_training.yaml')
query_config = load_config(PROJECT_DIR / 'configs/leafscan_test.yaml')
checkpoint_marker = require_existing_path(
    PROJECT_DIR / '.genus_checkpoint_path',
    'Выполните секцию 5: Locate The Genus Checkpoint On Drive.',
)
genus_checkpoint = require_existing_path(
    Path(checkpoint_marker.read_text(encoding='utf-8').strip()),
    'Выполните секцию 5: Locate The Genus Checkpoint On Drive.',
)

reference_records = apply_subset(
    filter_records_by_split(load_records(config['dataset']), 'train'),
    config['dataset'],
)
query_records = load_records(query_config['dataset'])
allowed_species = {record.species for record in query_records}
selected_reference_records = [record for record in reference_records if record.species in allowed_species]

genus_references = select_genus_references(
    selected_reference_records,
    references_per_genus=REFERENCES_PER_GENUS,
    seed=SPECIES_REFERENCE_SEED,
)
species_references = select_species_references(
    reference_records,
    references_per_species=SPECIES_REFERENCES_PER_LABEL,
    allowed_species=allowed_species,
)
refs_by_species = defaultdict(list)
for record in species_references:
    refs_by_species[record.species].append(record)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_config = config['model']
genus_model = build_siamese_network(BackboneSpec(name=model_config['backbone'], pretrained=False)).to(device)
species_model = build_siamese_network(BackboneSpec(name=model_config['backbone'], pretrained=False)).to(device)
genus_model.load_state_dict(torch.load(genus_checkpoint, map_location=device))
species_model.load_state_dict(torch.load(species_checkpoint, map_location=device))
genus_model.eval()
species_model.eval()

image_size = int(config['views']['global']['image_size'])
crop_size = int(config['views']['local']['crop_size'])
preprocessing = preprocessing_enabled(config)
print('Считаю эталонные эмбеддинги для S-CNN(A/B)...')
genus_embeddings = build_reference_embeddings(
    genus_model=genus_model,
    species_model=species_model,
    references=genus_references,
    image_size=image_size,
    crop_size=crop_size,
    preprocessing=preprocessing,
    device=device,
)
species_embeddings = build_reference_embeddings(
    genus_model=genus_model,
    species_model=species_model,
    references=species_references,
    image_size=image_size,
    crop_size=crop_size,
    preprocessing=preprocessing,
    device=device,
)
predictor = TwoStageSiamesePredictor(
    genus_model=genus_model,
    species_model=species_model,
    references=species_embeddings,
    genus_references=genus_embeddings,
    genus_candidates=SPECIES_GENUS_CANDIDATES,
    top_k=5,
    image_size=image_size,
    local_crop_size=crop_size,
    preprocessing=preprocessing,
    genus_score_mode=SPECIES_GENUS_SCORE_MODE,
    species_score_mode=SPECIES_SCORE_MODE,
    species_aggregation=SPECIES_AGGREGATION,
    device=str(device),
)

species_pair_candidates = []
for query in query_records:
    prediction = predictor.predict_many([query.image_path], top_k=1)[0]
    label = prediction.top_label
    if label is None or label.species == query.species:
        continue
    reference_options = refs_by_species.get(label.species, [])
    if not reference_options:
        continue
    reference = max(
        reference_options,
        key=lambda item: visual_similarity(query.image_path, item.image_path),
    )
    species_pair_candidates.append(
        {
            'left': query,
            'right': reference,
            'visual_similarity': visual_similarity(query.image_path, reference.image_path),
            'same_genus': int(label.genus == query.genus),
        }
    )

species_pair_candidates = sorted(
    species_pair_candidates,
    key=lambda item: (
        -item['same_genus'],
        0 if item['left'].genus in INTEREST_GENERA or item['right'].genus in INTEREST_GENERA else 1,
        -item['visual_similarity'],
    ),
)
release_cuda_model(genus_model)
release_cuda_model(species_model)
del predictor, genus_model, species_model

selected_species_pairs = choose_demo_pairs(
    species_pair_candidates,
    SPECIES_FRAME_COUNT * PAIRS_PER_FRAME,
    label_key='species',
    priority_getter=lambda item: (-item['same_genus'],),
)
print(f'Подобрано пар по видам: {len(selected_species_pairs)} из {len(species_pair_candidates)} кандидатов')
species_demo_frame_paths = render_demo_frames(
    selected_species_pairs,
    label_getter=lambda record: f'Вид: {short_species_name(record.species)}',
    output_prefix='species_demo_pairs_l1_l1_mean_R15',
    frame_count=SPECIES_FRAME_COUNT,
)
